In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "redis", "pandas", "matplotlib"])

## Dependencies

# Notebook 5 — Anomaly Analysis

**Technique:** Streaming anomaly detection post-hoc analysis

**Goal:** Analyse the anomaly events detected in real-time by `AnomalyDetector.java` to:
1. Count anomaly distribution by type, route, and vehicle
2. Show temporal clustering around the April 30 parade
3. Identify chronically malfunctioning GPS units
4. Verify that detected anomalies align with known disruptions

**Anomaly types detected in-stream:**
- `SPEED_EXCESS:N` — speed > 80 km/h (urban threshold)
- `GPS_JUMP:N.Nkm_in_Ns` — >500m movement in <30s (teleportation)

In [ ]:
%matplotlib inline
import redis
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
print(f'Redis connected. DB keys: {r.dbsize()}')

In [ ]:
# load event bất thường từ Redis, tối đa 500
raw = r.lrange('anomalies:recent', 0, -1)
print(f'Anomaly events in Redis: {len(raw)}')

anomalies = [json.loads(x) for x in raw]
adf = pd.DataFrame(anomalies)
if adf.empty:
    print('No anomalies recorded yet — run the producer for a while first.')
else:
    adf['datetime'] = pd.to_numeric(adf['datetime'], errors='coerce')
    adf['timestamp'] = adf['datetime'].apply(
        lambda ts: datetime.fromtimestamp(ts, tz=timezone.utc) if pd.notna(ts) else None
    )
    adf['anomaly_category'] = adf['anomalyType'].str.split(':').str[0]
    print(adf[['vehicle','anomalyType','speed','distanceFromPrevKm','timestamp']].head(10))

In [ ]:
if not adf.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # 1. phân phối loại bất thường
    adf['anomaly_category'].value_counts().plot(kind='bar', ax=axes[0], color='#e94560')
    axes[0].set_title('Anomaly Types')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=30)

    # 2. bất thường theo tuyến
    if 'routeNo' in adf.columns:
        adf['routeNo'].value_counts().head(15).plot(kind='bar', ax=axes[1], color='#4a90d9')
        axes[1].set_title('Anomalies by Route')
        axes[1].set_xlabel('')
        axes[1].tick_params(axis='x', rotation=45)

    # 3. bất thường theo thời gian
    if adf['timestamp'].notna().any():
        adf_time = adf.set_index('timestamp').resample('1H').size()
        adf_time.plot(ax=axes[2], color='#f5a623')
        axes[2].set_title('Anomalies per Hour')
        axes[2].set_xlabel('Time')
        axes[2].set_ylabel('Count')

    plt.tight_layout()
    plt.savefig('/tmp/anomaly_analysis.png', dpi=150)
    plt.show()

In [ ]:
if not adf.empty:
    # xe có nhiều bất thường — GPS có thể bị hỏng
    top_anomalous = adf.groupby('vehicle').size().sort_values(ascending=False).head(10)
    print('Top 10 vehicles by anomaly count (potential GPS hardware issues):')
    print(top_anomalous.to_string())

In [ ]:
if not adf.empty:
    # phân phối tốc độ cho anomaly vượt tốc
    speed_anomalies = adf[adf['anomaly_category'] == 'SPEED_EXCESS'].copy()
    speed_anomalies['speed'] = pd.to_numeric(speed_anomalies['speed'], errors='coerce')
    if len(speed_anomalies) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        speed_anomalies['speed'].hist(bins=30, ax=ax, color='#e94560', edgecolor='black')
        ax.axvline(80, color='orange', linestyle='--', label='80 km/h threshold')
        ax.set_title('Speed Distribution for SPEED_EXCESS Anomalies')
        ax.set_xlabel('Speed (km/h)')
        ax.legend()
        plt.tight_layout()
        plt.savefig('/tmp/speed_anomaly_dist.png', dpi=150)
        plt.show()